<a href="https://colab.research.google.com/github/jyizheng/my-study/blob/main/leetcode/Singleton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implement the singleton pattern with a twist. First, instead of storing one instance, store two instances. And in every even call of getInstance(), return the first instance and in every odd call of getInstance(), return the second instance.





``` cpp
#include <atomic>
#include <iostream>
#include <string>

class Dualton {
public:
    // 对外不可复制/移动，保证单例语义
    Dualton(const Dualton&) = delete;
    Dualton& operator=(const Dualton&) = delete;
    Dualton(Dualton&&) = delete;
    Dualton& operator=(Dualton&&) = delete;

    // 访问器：奇数次 -> second，偶数次 -> first
    static Dualton& getInstance() {
        // C++11 起，函数内静态局部变量的初始化是线程安全的
        static Dualton first("FIRST");
        static Dualton second("SECOND");

        static std::atomic<unsigned long long> calls{0};
        unsigned long long n = calls.fetch_add(1, std::memory_order_relaxed) + 1; // 1-based
        return (n % 2 == 0) ? first : second;
    }

    const std::string& label() const { return label_; }

private:
    explicit Dualton(std::string label) : label_(std::move(label)) {}
    std::string label_;
};

// 简单验证
int main() {
    for (int i = 1; i <= 6; ++i) {
        Dualton& inst = Dualton::getInstance();
        std::cout << "call #" << i
                  << " -> " << inst.label()
                  << " @" << &inst << '\n';
    }
}

```





Below the python implement

In [1]:
import threading

class Dualton:
    _first = None
    _second = None
    _lock = threading.Lock()
    _calls = 0

    def __init__(self, label):
        self.label = label

    @classmethod
    def get_instance(cls):
        # lazy init of the two instances
        if cls._first is None or cls._second is None:
            with cls._lock:
                if cls._first is None:
                    cls._first = Dualton("FIRST")
                if cls._second is None:
                    cls._second = Dualton("SECOND")

        with cls._lock:
            cls._calls += 1
            n = cls._calls  # 1-based
        return cls._first if n % 2 == 0 else cls._second

    def __repr__(self):
        return f"Dualton({self.label})"
